# VectorDB, Chroma로 문서 저장·검색 실습
- numpy 로 만든 검색기는 직관에는 좋지만 청크 1만개를 넘어가면 매번 행렬 곱이 무거워짐
- **Chroma** 는 오픈 소스 임베디드 벡터 DB로, 노트북에 그대로 띄워서 쓸 수 있고 로컬 디스크에 영속화도 됨


## 1. 환경 준비

## (1) 라이브러리 설치

처음 실행하는 환경이라면 아래 셀의 주석을 해제하고 실행합니다. 이미 설치되어 있다면 실행하지 않아도 됩니다.


In [ ]:
# 필요한 라이브러리 설치
# uv add -qU langchain langchain-chroma langchain-openai langchain-text-splitters python-dotenv chromadb


## (2) API Key 설정

API 키는 코드에 직접 작성하지 않습니다. 권장 방식은 `.env` 파일에 저장하는 것입니다.

```text
# OpenAI를 사용할 때
OPENAI_API_KEY=sk-...

# Gemini를 사용할 때
GOOGLE_API_KEY=...

```


In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

print("OPENAI_API_KEY:", "있음" if os.getenv("OPENAI_API_KEY") else "없음")
print("GOOGLE_API_KEY:", "있음" if os.getenv("GOOGLE_API_KEY") else "없음")

OPENAI_API_KEY: 있음
GOOGLE_API_KEY: 있음


## 2. VectorDB 핵심 개념

| 개념 | 의미 |
|---|---|
| Collection | 관련 문서 벡터를 담는 저장 단위 |
| Document | `page_content` 와 `metadata` 를 가진 LangChain 문서 객체 |
| Embedding function | 텍스트를 벡터로 바꾸는 함수 또는 모델 |
| Metadata | `source`, `section`, `owner` 같은 필터링·출처 정보 |
| ID | 문서를 갱신·삭제할 때 사용하는 고유 식별자 |
| Persist directory | vectorDB 데이터를 로컬 디스크에 저장하는 경로 |


## 3. 실습 문서 만들기

- 작은 사내 규정 문서를 `Document` 형태로 준비
- 실무에서는 PDF/Markdown/HTML 로더가 만든 문서도 같은 형태로 vectorDB 에 저장

In [49]:
from langchain_core.documents import Document

# Document 객체로 추가 (텍스트 + 메타데이터)
policy_docs = [
    Document(
        page_content="신규 입사자는 입사 후 7일 이내에 보안 교육을 이수해야 하며, 교육 미이수 시 사내 시스템 접근 권한이 제한될 수 있다.",
        metadata={"source": "hr_policy.md", "section": "보안교육", "owner": "HR", "version": "2026.06"},
    ),
    Document(
        page_content="법인카드 사용 내역은 결제일 기준 5영업일 이내에 영수증과 함께 경비 처리 시스템에 등록해야 한다.",
        metadata={"source": "expense_policy.md", "section": "경비처리", "owner": "Finance", "version": "2026.06"},
    ),
    Document(
        page_content="개인정보가 포함된 문서는 외부 공유 전에 반드시 비식별 처리해야 하며, 고객명과 연락처는 마스킹 대상에 포함된다.",
        metadata={"source": "security_guide.md", "section": "개인정보", "owner": "Security", "version": "2026.06"},
    ),
    Document(
        page_content="장애 보고서는 발생 시각, 영향 범위, 임시 조치, 재발 방지 대책을 포함하여 서비스 복구 후 24시간 이내에 작성해야 한다.",
        metadata={"source": "dev_standards.md", "section": "장애보고", "owner": "Engineering", "version": "2026.06"},
    ),
    Document(
        page_content="재택근무 신청은 최소 하루 전까지 근태 시스템에서 등록해야 하며, 팀장의 승인을 받은 뒤 근무 장소를 명시해야 한다.",
        metadata={"source": "hr_policy.md", "section": "재택근무", "owner": "HR", "version": "2026.06"},
    ),
    Document(
        page_content="회의실 예약은 회의 시작 최소 2시간 전까지 완료해야 하며, 3시간을 초과하는 회의는 조직장 승인이 필요하다.",
        metadata={"source": "office_guide.md", "section": "회의실", "owner": "Admin", "version": "2026.06"},
    ),
]

doc_ids = [f"policy-{i:02d}" for i in range(len(policy_docs))]

for doc_id, doc in zip(doc_ids, policy_docs):
    print(f"{doc_id} | {doc.metadata['section']} | {doc.page_content[:45]}...")


policy-00 | 보안교육 | 신규 입사자는 입사 후 7일 이내에 보안 교육을 이수해야 하며, 교육 미이수 시 ...
policy-01 | 경비처리 | 법인카드 사용 내역은 결제일 기준 5영업일 이내에 영수증과 함께 경비 처리 시스템...
policy-02 | 개인정보 | 개인정보가 포함된 문서는 외부 공유 전에 반드시 비식별 처리해야 하며, 고객명과 ...
policy-03 | 장애보고 | 장애 보고서는 발생 시각, 영향 범위, 임시 조치, 재발 방지 대책을 포함하여 서...
policy-04 | 재택근무 | 재택근무 신청은 최소 하루 전까지 근태 시스템에서 등록해야 하며, 팀장의 승인을 ...
policy-05 | 회의실 | 회의실 예약은 회의 시작 최소 2시간 전까지 완료해야 하며, 3시간을 초과하는 회...


## 4. Chroma 첫 사용, 메모리 모드
- 메모리 모드는 노트북을 종료하면 사라지는 임시 저장소
- 빠른 실험과 수업용 데모에 적합함

In [3]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [50]:
from langchain_chroma import Chroma

vectorstore = Chroma(
    collection_name='company_policy_memory',        # 관계형 DB의 테이블 이름과 비슷한 개념
    embedding_function=embeddings
)

# policy_docs 문서들을 ChromaDB에 저장
# ids=doc_ids : 각 문서에 부여한 고유 ID 목록. 나중에 문서를 추가하거나 중복 저장을 피할 때 유용
stored_ids = vectorstore.add_documents(policy_docs, ids=doc_ids)

In [51]:
print(f"저장 문서 수: {len(stored_ids)}")
print(f"Chroma count: {vectorstore._collection.count()}")

저장 문서 수: 6
Chroma count: 397


## 5. 유사도 검색, `similarity_search`
- 질문을 임베딩한 뒤 collection 안의 문서 벡터와 비교해 가까운 문서를 반환


In [7]:
results = vectorstore.similarity_search('법인카드 영수증은 언제까지 등록해야 되나요?', k=3)

for doc in results:
    print(f"[{doc.metadata['section']}] {doc.page_content}")


[경비처리] 법인카드 사용 내역은 결제일 기준 5영업일 이내에 영수증과 함께 경비 처리 시스템에 등록해야 한다.
[재택근무] 재택근무 신청은 최소 하루 전까지 근태 시스템에서 등록해야 하며, 팀장의 승인을 받은 뒤 근무 장소를 명시해야 한다.
[회의실] 회의실 예약은 회의 시작 최소 2시간 전까지 완료해야 하며, 3시간을 초과하는 회의는 조직장 승인이 필요하다.


## 6. 점수까지 보기, `similarity_search_with_score`
- Chroma 의 score 는 기본적으로 **distance** 인데, 값이 작을수록 더 가까움.
- 다른 retriever 의 relevance score 는 값이 클수록 관련도가 높은 경우가 있어 방향을 구분해야 함


In [8]:
# cosine distance = 1 - cosine similarity
scored_results = vectorstore.similarity_search_with_score('장애가 나면 보고서에 무엇을 써야 하나요?', k=3)

for doc, distance in scored_results:
    print(f"distance={distance:.3f} | [{doc.metadata['section']}] {doc.page_content[:80]}")


distance=1.000 | [장애보고] 장애 보고서는 발생 시각, 영향 범위, 임시 조치, 재발 방지 대책을 포함하여 서비스 복구 후 24시간 이내에 작성해야 한다.
distance=1.771 | [보안교육] 신규 입사자는 입사 후 7일 이내에 보안 교육을 이수해야 하며, 교육 미이수 시 사내 시스템 접근 권한이 제한될 수 있다.
distance=1.786 | [개인정보] 개인정보가 포함된 문서는 외부 공유 전에 반드시 비식별 처리해야 하며, 고객명과 연락처는 마스킹 대상에 포함된다.


## 7. 메타데이터 필터 검색
- 문서 내용 검색과 별개로 `metadata` 조건을 걸 수 있음
- Chroma 의 `filter` 인자에 `{"메타필드": "값"}` 를 넘기면 메타데이터 조건 검색


In [9]:
# Finance 담당 문서 안에서만 검색
finance_results = vectorstore.similarity_search(
    '증빙 제출 기한',
    k=3,
    filter={'owner': 'Finance'}
)


for doc in finance_results:
    print(f"[{doc.metadata['owner']}/{doc.metadata['section']}] {doc.page_content}")

[Finance/경비처리] 법인카드 사용 내역은 결제일 기준 5영업일 이내에 영수증과 함께 경비 처리 시스템에 등록해야 한다.


In [18]:
# 여러 owner 를 한 번에 필터링
# 여기서는 owner가 "HR" 또는 "Admin"인 문서만 검색 대상이 됨
hr_or_admin_results = vectorstore.similarity_search(
    # 검색 질문 또는 검색어
    '근무 장소나 회의 예약 규정',
    # 유사도가 높은 문서 3개만 반환
    k=3,
    # metadata 필터 조건
    # owner 값이 ["HR", "Admin"] 중 하나인 문서만 검색
    filter = {'owner':{'$in':['HR','Admin']}}
)
# 검색 결과 출력
for doc in hr_or_admin_results:
    # metadata에서 owner와 section을 꺼내고,
    # 해당 문서의 본문(page_content)을 함께 출력
    print(f"[{doc.metadata['owner']}/{doc.metadata['section']}] {doc.page_content}")


[Admin/회의실] 회의실 예약은 회의 시작 최소 2시간 전까지 완료해야 하며, 3시간을 초과하는 회의는 조직장 승인이 필요하다.
[HR/재택근무] 재택근무 신청은 최소 하루 전까지 근태 시스템에서 등록해야 하며, 팀장의 승인을 받은 뒤 근무 장소를 명시해야 한다.
[HR/보안교육] 신규 입사자는 입사 후 7일 이내에 보안 교육을 이수해야 하며, 교육 미이수 시 사내 시스템 접근 권한이 제한될 수 있다.


## 8. 문서 추가, 갱신, 삭제
- 운영 환경에서는 문서가 계속 바뀜
- vectorDB 에도 문서 ID 기준의 추가·갱신·삭제 전략이 필요함


In [10]:
# 추가
new_id = vectorstore.add_texts(
    texts=['외부 교육비는 교육 종료 후 10영업일 이내에 수료증과 영수증을 함께 제출해야 한다.'],
    metadatas=[{"source": "expense_policy.md", "section": "교육비", "owner": "Finance", "version": "2026.06"}])
print(f"새 ID: {new_id}")

새 ID: ['4b1722fb-1bbe-44d6-b085-73c0154517a8']


In [11]:
# 확인
results = vectorstore.similarity_search('외부 교육비?', k=2)
for d in results:
    print(f"  {d.page_content}")

  외부 교육비는 교육 종료 후 10영업일 이내에 수료증과 영수증을 함께 제출해야 한다.
  신규 입사자는 입사 후 7일 이내에 보안 교육을 이수해야 하며, 교육 미이수 시 사내 시스템 접근 권한이 제한될 수 있다.


In [12]:
# 삭제
vectorstore.delete(ids=new_id)
print('삭제 완료')

# 확인
results = vectorstore.similarity_search('외부 교육비?', k=2)
for d in results:
    print(f"  {d.page_content}")

삭제 완료
  신규 입사자는 입사 후 7일 이내에 보안 교육을 이수해야 하며, 교육 미이수 시 사내 시스템 접근 권한이 제한될 수 있다.
  법인카드 사용 내역은 결제일 기준 5영업일 이내에 영수증과 함께 경비 처리 시스템에 등록해야 한다.


## 9. 영속화, 로컬 디스크에 저장

- `persist_directory` 를 지정하면 Chroma collection 이 로컬 폴더에 저장됨
- 노트북을 다시 열어도 같은 경로로 불러올 수 있음

In [13]:
import shutil
from pathlib import Path

# ChromaDB 데이터를 저장할 로컬 폴더 경로 지정
PERSIST_DIR = Path("./.chroma_company_policy")

# 기존에 같은 폴더가 있으면 삭제
# 실습을 매번 깨끗한 상태에서 다시 시작하기 위한 코드
shutil.rmtree(PERSIST_DIR, ignore_errors=True)

# 디스크에 저장되는 Chroma 벡터 DB 생성
persisted_store = Chroma(
    # Chroma 안에서 사용할 컬렉션 이름
    # 하나의 DB 안에 여러 컬렉션을 둘 수 있음
    collection_name="company_policy_disk",
    # 문서를 벡터로 변환할 때 사용할 임베딩 모델
    embedding_function=embeddings,
    # ChromaDB 데이터를 실제 파일로 저장할 디렉터리
    # 이 옵션이 있으면 메모리에서만 쓰지 않고 디스크에 저장됨
    persist_directory=str(PERSIST_DIR),          
)
# policy_docs 문서들을 ChromaDB에 추가
# ids는 각 문서에 부여할 고유 ID 목록
persisted_store.add_documents(policy_docs,
                              ids=doc_ids)
# 현재 컬렉션에 저장된 문서 개수 확인
print("저장 count:", persisted_store._collection.count())

# 같은 persist_directory와 collection_name으로 Chroma를 다시 생성
# 이미 디스크에 저장된 데이터를 다시 불러오는 역할
reloaded_store = Chroma(
    collection_name="company_policy_disk",
    embedding_function=embeddings,
    persist_directory=str(PERSIST_DIR),
)
# 다시 로드한 ChromaDB에 문서가 그대로 남아 있는지 확인
print("다시 로드한 count:", reloaded_store._collection.count())


저장 count: 6
다시 로드한 count: 6


## 10. Retriever 로 변환하기
- RAG 체인에 연결할 때는 `similarity_search()`를 직접 호출하기보다 `as_retriever()`로 변환해 두면 LCEL 체인에 끼우기 쉬움


In [14]:
retriever = vectorstore.as_retriever(
    search_type = 'similarity',
    search_kwargs = {'k':3}
)

retrieved_docs = retriever.invoke("개인정보 문서를 외부에 공유하려면 무엇을 해야 하나요?")
for doc in retrieved_docs:
    print(f"[{doc.metadata['owner']}/{doc.metadata['section']}] {doc.page_content}")


[Security/개인정보] 개인정보가 포함된 문서는 외부 공유 전에 반드시 비식별 처리해야 하며, 고객명과 연락처는 마스킹 대상에 포함된다.
[Engineering/장애보고] 장애 보고서는 발생 시각, 영향 범위, 임시 조치, 재발 방지 대책을 포함하여 서비스 복구 후 24시간 이내에 작성해야 한다.
[Finance/경비처리] 법인카드 사용 내역은 결제일 기준 5영업일 이내에 영수증과 함께 경비 처리 시스템에 등록해야 한다.


In [15]:
finance_retriever = vectorstore.as_retriever(
    search_type = 'similarity',
    search_kwargs = {'k':3, 'filter': {'owner':'Finance'}}
)

retrieved_docs = finance_retriever.invoke("영수증과 증빙은 언제 올려야하나요?")
for doc in retrieved_docs:
    print(f"[{doc.metadata['owner']}/{doc.metadata['section']}] {doc.page_content}")


[Finance/경비처리] 법인카드 사용 내역은 결제일 기준 5영업일 이내에 영수증과 함께 경비 처리 시스템에 등록해야 한다.


## 11. 정리

- Chroma 는 로컬에서 바로 사용할 수 있는 vectorDB 입니다.
- `add_documents()` 로 문서와 메타데이터를 저장합니다.
- `similarity_search()` 는 유사 문서를 반환하고, `_with_score()` 는 distance 까지 반환합니다.
- `filter` 로 source, section, owner 같은 metadata 조건을 걸 수 있습니다.
- `persist_directory` 로 디스크에 저장하면 재실행 후에도 다시 로드할 수 있습니다.
- `as_retriever()` 로 바꾸면 RAG 체인에 연결하기 쉽습니다.


## [실습]

1. `k=1`, `k=3`, `k=5` 로 검색 결과를 비교합니다.
2. `filter={"section": "재택근무"}` 조건으로 재택근무 질문을 검색합니다.
3. `owner` 가 `HR` 또는 `Security` 인 문서만 검색하도록 `$in` 필터를 작성합니다.
4. 새 정책 문서 3개를 추가하고 검색 결과가 바뀌는지 확인합니다.
5. 05 차시의 `RAG_PROMPT`와 연결해 Chroma 기반 RAG 체인을 만듭니다.


In [26]:
for k in [1, 3, 5]:
    hr_or_security_results = vectorstore.similarity_search(
        "근무 장소나 회의 예약 규정",
        k=k,
        filter={
    "$and": [
        {"section": "재택근무"},
        {"owner": {"$in": ["HR", "Security"]}}
    ]
}
    )

    print(f"\n===== k = {k} =====")

    for doc in hr_or_security_results:
        print(f"[{doc.metadata['owner']}/{doc.metadata['section']}] {doc.page_content}")


===== k = 1 =====
[HR/재택근무] 재택근무 신청은 최소 하루 전까지 근태 시스템에서 등록해야 하며, 팀장의 승인을 받은 뒤 근무 장소를 명시해야 한다.

===== k = 3 =====
[HR/재택근무] 재택근무 신청은 최소 하루 전까지 근태 시스템에서 등록해야 하며, 팀장의 승인을 받은 뒤 근무 장소를 명시해야 한다.

===== k = 5 =====
[HR/재택근무] 재택근무 신청은 최소 하루 전까지 근태 시스템에서 등록해야 하며, 팀장의 승인을 받은 뒤 근무 장소를 명시해야 한다.


In [40]:
# 추가
new_id2 = vectorstore.add_texts(
    texts=["회의실 예약은 회의 시작 최소 2시간 전까지 완료해야 하며, 3시간을 초과하는 회의는 조직장 승인이 필요하다."],
    metadatas=[{"source": "office_guide.md", "section": "회의실", "owner": "Admin", "version": "2026.06"}])
new_id3 = vectorstore.add_texts(
    texts=["장애 보고서는 발생 시각, 영향 범위, 임시 조치, 재발 방지 대책을 포함하여 서비스 복구 후 24시간 이내에 작성해야 한다."],
    metadatas=[{"source": "dev_standards.md", "section": "장애보고", "owner": "Engineering", "version": "2026.06"}])
new_id4 = vectorstore.add_texts(
    texts=["개인정보가 포함된 문서는 외부 공유 전에 반드시 비식별 처리해야 하며, 고객명과 연락처는 마스킹 대상에 포함된다."],
    metadatas=[{"source": "security_guide.md", "section": "개인정보", "owner": "Security", "version": "2026.06"}])
print(f"새 ID: {new_id2}")
print(f"새 ID: {new_id3}")
print(f"새 ID: {new_id4}")

새 ID: ['ebeebff6-51a2-4929-a9bf-c58c98b2b051']
새 ID: ['0e1a11c4-a3aa-4146-bddd-4e113ffbcd09']
새 ID: ['800006c5-9235-41da-9d38-8fcdeb1df261']


In [44]:
for k in [1, 3, 5]:
    hr_or_security_results = vectorstore.similarity_search(
        "근무 장소나 회의 예약 규정",
        k=k,
        filter={
    "$and": [
        {"section": {'$in' : ["재택근무", "회의실", "장애보고", "개인정보"]}},
        {"owner": {'$in' : ["HR", "Admin", "Engineering", "Security"]}}
    ]
}
    )

    print(f"\n===== k = {k} =====")

    for doc in hr_or_security_results:
        print(f"[{doc.metadata['owner']}/{doc.metadata['section']}] {doc.page_content}")


===== k = 1 =====
[Admin/회의실] 회의실 예약은 회의 시작 최소 2시간 전까지 완료해야 하며, 3시간을 초과하는 회의는 조직장 승인이 필요하다.

===== k = 3 =====
[Admin/회의실] 회의실 예약은 회의 시작 최소 2시간 전까지 완료해야 하며, 3시간을 초과하는 회의는 조직장 승인이 필요하다.
[Admin/회의실] 회의실 예약은 회의 시작 최소 2시간 전까지 완료해야 하며, 3시간을 초과하는 회의는 조직장 승인이 필요하다.
[HR/재택근무] 재택근무 신청은 최소 하루 전까지 근태 시스템에서 등록해야 하며, 팀장의 승인을 받은 뒤 근무 장소를 명시해야 한다.

===== k = 5 =====
[Admin/회의실] 회의실 예약은 회의 시작 최소 2시간 전까지 완료해야 하며, 3시간을 초과하는 회의는 조직장 승인이 필요하다.
[Admin/회의실] 회의실 예약은 회의 시작 최소 2시간 전까지 완료해야 하며, 3시간을 초과하는 회의는 조직장 승인이 필요하다.
[HR/재택근무] 재택근무 신청은 최소 하루 전까지 근태 시스템에서 등록해야 하며, 팀장의 승인을 받은 뒤 근무 장소를 명시해야 한다.
[Admin/회의실] 회
[Admin/회의실] 회


In [38]:
finance_retriever = vectorstore.as_retriever(
    search_type = 'similarity',
    search_kwargs = {'k':1, 'filter' : {"owner": {'$in' : ["HR", "Admin", "Engineering", "Security"]}}}
)

In [ ]:
import numpy as np

chunk_vectors = np.array(embeddings.embed_documents(vectorstore))
chunk_vectors_n = chunk_vectors / np.linalg.norm(chunk_vectors,
                                                 axis=-1,
                                                 keepdims=True)

def retrieve(query: str, k: int = 3):
    # 6-1. 질문을 임베딩 벡터로 변환
    q_vec = embeddings.embed_query(query)
    # 6-2. 질문 벡터도 정규화
    q_vec_n = np.array(q_vec) / np.linalg.norm(q_vec)
    # 6-3. 질문 벡터와 모든 청크 벡터의 유사도 계산
    sims = chunk_vectors_n @ q_vec_n
    # 6-4. 유사도가 높은 순서대로 상위 k개 인덱스 선택
    top_idx = np.argsort(sims)[::-1][:k]
    # 6-5. 선택된 인덱스에 해당하는 원본 청크 반환
    return [vectorstore[i] for i in top_idx]